# Week 9 · Notebook 2 — Same Prompt, Claude vs Gemini (+ Refinement)
**CSCI 250 · Fall 2026**

Run **one prompt** on **both providers**, then run an **iterative refinement** loop and check JSON parsing on a small eval set.

> **Keys:** Colab 🔑 *Secrets* + `userdata.get('NAME')`; locally use env vars. **Never commit keys.**

## 0. Install + keys

In [ ]:
!pip -q install anthropic google-generativeai

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    pass
print('Anthropic key set:', bool(os.environ.get('ANTHROPIC_API_KEY')))
print('Gemini key set:', bool(os.environ.get('GEMINI_API_KEY')))

## 1. Wrap both providers behind one interface
Same prompting patterns; only the call surface differs.

In [ ]:
import anthropic, google.generativeai as genai

client = anthropic.Anthropic()
genai.configure(api_key=os.environ.get('GEMINI_API_KEY', ''))

def claude(prompt, system=None, temperature=0.0, max_tokens=400):
    kw = dict(model='claude-sonnet-4-6', max_tokens=max_tokens,
              temperature=temperature,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

def gemini(prompt, system=None, temperature=0.0, max_tokens=400):
    model = genai.GenerativeModel('gemini-2.5-flash',
                                  system_instruction=system)
    r = model.generate_content(prompt, generation_config={
        'temperature': temperature, 'max_output_tokens': max_tokens})
    return r.text

## 2. Same prompt → both providers
Note: Claude uses `system=`, Gemini uses `system_instruction=`.

In [ ]:
SYS = 'You are a concise teaching assistant for an intro AI course.'
PROMPT = 'In 3 bullet points, explain few-shot prompting to a beginner.'
for name, fn in [('CLAUDE', claude), ('GEMINI', gemini)]:
    try:
        print(f'=== {name} ===')
        print(fn(PROMPT, system=SYS), '\n')
    except Exception as e:
        print(name, 'error:', e, '\n')

## 3. Iterative refinement on a real task
Task: extract structured fields from messy product reviews. We start zero-shot, then refine to **few-shot + JSON + role + delimiters** and test on an eval set.

In [ ]:
import json

EVAL = [
    'Acme X100 — battery dies in 2 hrs, super disappointing.',
    'The Globex Pro is pricey but the camera is stunning.',
    'Initech Mini: nothing special, just an average phone.',
]

def extract_json(text):
    t = text.strip()
    if t.startswith('```'):
        t = t.split('```')[1]
        if t.startswith('json'): t = t[4:]
    return json.loads(t.strip())

# v1: vague zero-shot (often wrong format)
def v1(review):
    return claude(f'Get the product and how the reviewer felt: {review}')

# v2: refined — role + few-shot + JSON + delimiters
ROLE = 'You are a precise data extractor. Output JSON only, no prose.'
def v2(review):
    p = (
      'Extract keys product, sentiment (positive/negative/neutral) as JSON.\n'
      'Example -> Review: "Foo A1 is amazing" => '
      '{"product": "Foo A1", "sentiment": "positive"}\n'
      f'Review between tags:\n<r>{review}</r>'
    )
    return claude(p, system=ROLE)

print('--- v1 (zero-shot) ---')
try:
    for r in EVAL: print(v1(r))
except Exception as e:
    print('Set ANTHROPIC_API_KEY to run:', e)

In [ ]:
print('--- v2 (refined) — checking all 3 parse as JSON ---')
try:
    ok = 0
    for r in EVAL:
        data = extract_json(v2(r))
        print(data); ok += 1
    print(f'{ok}/{len(EVAL)} parsed cleanly.')
except Exception as e:
    print('Set ANTHROPIC_API_KEY (or check JSON) to run:', e)

## 4. A4 deliverable
1. Replace the EVAL list with **your own** 3 inputs for a task you choose.
2. Refine your prompt until all 3 parse with `json.loads`.
3. Run your final prompt on **Gemini** too (swap `claude` for `gemini`) and note differences.
4. Write ~150 words **before/after**: which changes helped most, and why?

In [ ]:
# your task + reflection notes:
